# Day 1 · Hands-on 2 — Confidence intervals
**About 30 min · work in pairs** (one person drives the keyboard, the other reads, predicts and questions — swap halfway)


A 95% confidence interval comes from a **procedure** that captures the true value in 95% of studies. Any single interval either contains the truth or it doesn't. Because we know the true mean of our GBSG2 "population", we can check.



> **Two AI roles — keep them apart**
> - 🧮 **Gemini in Colab = calculator.** Ask it to write, change, fix or explain *code*, including new simulations.
> - 🎓 **Claude / ChatGPT (other tab) = tutor.** Use it to explain ideas, quiz you and critique your own work. Ask for its reasoning, and check one claim yourself in the notebook.


**How this notebook works:** run cells from top to bottom. Each code cell has exactly one line marked `# 🔧 CHANGE THIS` — that is the only thing you need to edit. Lines starting with 📋 are results to enter in the shared sheet.

In [ ]:
# ▶️ Run this cell first (about 20 seconds). Nothing to change here.
%pip install -q lifelines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from lifelines.datasets import load_gbsg2

df = load_gbsg2()
print(f"✅ Loaded GBSG2: {len(df)} patients, {df.shape[1]} variables")

## A. One study, one interval
### 🧭 Concept
From one sample we compute mean ± t × SE. In real life you **cannot** check whether your interval caught the truth — here you can.

In [ ]:
# 🔧 CHANGE THIS: number of patients in your study
n = 30

rng = np.random.default_rng()
population = df["tsize"].to_numpy()
true_mean = population.mean()

sample = rng.choice(population, size=n, replace=True)
mean = sample.mean()
se = sample.std(ddof=1) / np.sqrt(n)
t_crit = stats.t.ppf(0.975, df=n - 1)
low, high = mean - t_crit * se, mean + t_crit * se

print(f"Study mean: {mean:.1f} mm      95% CI: [{low:.1f}, {high:.1f}] mm")
print(f"True mean:  {true_mean:.1f} mm")
print("Did this interval capture the truth?", "✅ yes" if low <= true_mean <= high else "❌ no")

## B. 100 studies, 100 intervals
### 🧭 Concept
Each line below is one study's 95% CI. Red lines missed the true mean.

✍️ **Predict first:** out of 100 intervals, how many will be red?

Use your **pair number** as the seed so that every pair gets different studies.

In [ ]:
# 🔧 CHANGE THIS: your pair number (any whole number)
seed = 1

n = 30                       # patients per study (keep fixed for the shared sheet)
n_studies = 100
rng = np.random.default_rng(seed)
population = df["tsize"].to_numpy()
true_mean = population.mean()

samples = rng.choice(population, size=(n_studies, n), replace=True)
means = samples.mean(axis=1)
ses = samples.std(axis=1, ddof=1) / np.sqrt(n)
t_crit = stats.t.ppf(0.975, df=n - 1)
lows, highs = means - t_crit * ses, means + t_crit * ses
covered = (lows <= true_mean) & (true_mean <= highs)

plt.figure(figsize=(6, 8))
for i in range(n_studies):
    plt.plot([lows[i], highs[i]], [i, i], color="steelblue" if covered[i] else "red", linewidth=1.5)
plt.axvline(true_mean, color="black", linestyle="--", label="True mean")
plt.xlabel("Tumour size (mm)"); plt.ylabel("Study"); plt.legend(loc="lower right")
plt.title(f"100 studies of n = {n}: {covered.sum()} intervals caught the true mean")
plt.show()

print("📋 For the shared sheet:")
print(f"   seed = {seed}   |   intervals containing the true mean = {covered.sum()} / {n_studies}")

## C. What does the confidence level buy you?
### 🧭 Concept
A higher confidence level catches the truth more often, but costs you something. Find out what by trying 0.80, 0.90, 0.95 and 0.99.

In [ ]:
# 🔧 CHANGE THIS: confidence level (0.80, 0.90, 0.95, 0.99 ...)
conf_level = 0.95

n, n_studies = 30, 5000
rng = np.random.default_rng()
population = df["tsize"].to_numpy()
true_mean = population.mean()

samples = rng.choice(population, size=(n_studies, n), replace=True)
means = samples.mean(axis=1)
ses = samples.std(axis=1, ddof=1) / np.sqrt(n)
t_crit = stats.t.ppf(1 - (1 - conf_level) / 2, df=n - 1)
covered = np.abs(means - true_mean) <= t_crit * ses

print(f"Confidence level {conf_level:.0%}:")
print(f"   intervals catching the true mean: {covered.mean():.1%}")
print(f"   average interval width:           {np.mean(2 * t_crit * ses):.1f} mm")

## D. ⭐ If you have time: when the promise breaks
### 🧭 Concept
The t-interval assumes the sample mean is roughly normal. Tumour sizes are skewed. Try n = 5, 10, 30 and 100: does a "95%" interval still deliver 95%?

In [ ]:
# 🔧 CHANGE THIS: number of patients per study
n = 5

n_studies = 20000
rng = np.random.default_rng()
population = df["tsize"].to_numpy()
true_mean = population.mean()

samples = rng.choice(population, size=(n_studies, n), replace=True)
means = samples.mean(axis=1)
ses = samples.std(axis=1, ddof=1) / np.sqrt(n)
t_crit = stats.t.ppf(0.975, df=n - 1)
coverage = np.mean(np.abs(means - true_mean) <= t_crit * ses)
print(f"n = {n}: a nominal 95% interval actually caught the true mean {coverage:.1%} of the time")

### ✅ Concept check — which statements are correct?
Your study gives a 95% CI for mean tumour size of [26.1, 33.4] mm.
1. There is a 95% probability that the true mean lies between 26.1 and 33.4 mm.
2. If we repeated the study many times, about 95% of the intervals would contain the true mean.
3. 95% of patients have tumours between 26.1 and 33.4 mm.
4. Values inside the interval are reasonably compatible with the data; values far outside are not.

<details><summary>Show answers</summary>

**2** and **4** are correct. **1** is the classic misreading: once computed, this interval either contains the true mean or not; the 95% describes the method. **3** confuses the CI for the *mean* with the spread of *patients* (that is what the SD describes).
</details>

## 🤖 Go deeper with AI
Three ways to keep learning about confidence intervals: now, if you finish early, or after the workshop.

**🎓 Explain and quiz me** (Claude / ChatGPT):
> *Explain what a 95% confidence interval means using the idea of many repeated studies. Then give me 4 statements about one interval, one at a time, and ask me whether each is correct.*

**🧪 See it in a simulation** (🧮 Gemini in Colab, in a new cell at the end of this notebook):
> *Simulate 1,000 studies of 10 patients from the GBSG2 tumour sizes, compute a 95% t-interval for each, and count how many contain the true mean. Repeat with 50 patients per study.*

**🔬 Apply it to my work** (Claude / ChatGPT; replace the brackets with your own details):
> *Here are two results from my experiments with their 95% CIs: [paste]. What can I conclude about the difference between them, and what can't I conclude from whether the intervals overlap?*

✔ **Check one claim yourself:** Coverage should be close to 95% with 50 patients and a little lower with 10, because tumour sizes are skewed (section D).

## 💬 Discuss with your partner
- Why can we check coverage here but never in a real study?
- Section C showed a trade-off. Which confidence level would you choose for a pilot study, and which for a pivotal trial?
- In section D, coverage fell short with small samples of skewed data. What could you do about it in your own small experiments?